# 🍲 Tarif Defteri

Bu projede kendi tarif defterimi hazırladım. Tarifleri Python'da sınıflarla tuttum, sonra bunlardan bir HTML arayüz oluşturdum.

**Defterin bölümleri:**
1. `Tarif` sınıfı: tek bir tarifin bilgileri
2. `TarifDefteri` sınıfı: bütün tarifleri tutan, arama ve filtreleme yapan sınıf
3. Tariflerin deftere eklenmesi
4. Saklama süreleri: her yemeğin buzdolabında ortalama kaç gün dayandığı
5. `Buzdolabi` sınıfı: dolaptaki malzemelere göre yapılabilecek tarifleri bulan sınıf
6. `YemekGunlugu` sınıfı: her gün yediklerimizi kaydedip haftalık listeleyen sınıf
7. Arayüzün stil (CSS) ve çalışma (JavaScript) kodları
8. `Arayuz` sınıfı: HTML sayfayı oluşturup gösteren sınıf

Arayüzde üç sekme var: **📖 Tarifler**, **🧊 Buzdolabım** ve **📅 Yediklerim**.
Buzdolabı ve günlük bilgileri tarayıcıda saklanıyor, yani sayfa kapanıp açılınca kaybolmuyor.

Hücreleri sırayla çalıştırmak yeterli (veya *Çalışma zamanı → Tümünü çalıştır*).

In [ ]:
import json
import re
from datetime import date, timedelta
from IPython.display import HTML, display

## 1. Tarif sınıfı
Her tarifin adı, kaç kişilik olduğu, süreleri, kategorisi, malzemeleri ve yapılışı bu sınıfta duruyor.
Saklama süresi ve buzdolabı için kullanılan malzeme listesi tarifler eklendikten sonra dolduruluyor (4. ve 5. bölüm).

In [ ]:
# tarif resimleri bu siteden geliyor, her tarifte sadece adresin devamını yazıyorum
RESIM_ADRESI = "https://i.nefisyemektarifleri.com/"


def sure_yazisi(en_az, en_cok):
    # 3, 4 -> "3-4 gün" / 60, 90 -> "2-3 ay"
    if en_az >= 30:
        return f"{en_az // 30}-{en_cok // 30} ay"
    if en_az == en_cok:
        return f"{en_az} gün"
    return f"{en_az}-{en_cok} gün"


class Tarif:
    def __init__(self, ad, kisi, hazirlik, pisirme, kategori, emoji, resim, malzemeler, yapilis):
        self.ad = ad
        self.kisi = kisi
        self.hazirlik = hazirlik
        self.pisirme = pisirme
        self.kategori = kategori
        self.emoji = emoji
        self.resim = resim
        self.malzemeler = malzemeler
        self.yapilis = yapilis
        # bu ikisi sonradan dolduruluyor
        self.saklama = None
        self.anahtar_malzemeler = []

    def resim_linki(self):
        if self.resim == "":
            return ""
        return RESIM_ADRESI + self.resim

    def saklama_ekle(self, en_az_gun, en_cok_gun, aciklama):
        self.saklama = {"en_az": en_az_gun, "en_cok": en_cok_gun, "not": aciklama}

    def saklama_yazisi(self):
        if self.saklama is None:
            return "Bilgi yok"
        return sure_yazisi(self.saklama["en_az"], self.saklama["en_cok"])

    def sozluge_cevir(self):
        # html tarafında kullanabilmek için tarifi sözlüğe çeviriyorum
        return {
            "ad": self.ad,
            "kisi": self.kisi,
            "hazirlik": self.hazirlik,
            "pisirme": self.pisirme,
            "kategori": self.kategori,
            "emoji": self.emoji,
            "resim": self.resim_linki(),
            "malzemeler": self.malzemeler,
            "yapilis": self.yapilis,
            "saklama": self.saklama,
            "anahtarlar": self.anahtar_malzemeler,
        }

    def __str__(self):
        return f"{self.emoji} {self.ad} ({self.kisi} kişilik)"

## 2. TarifDefteri sınıfı
Tarifleri bir listede tutuyor. Tarif ekleme, isme göre arama ve kategoriye göre filtreleme yapabiliyor.

In [ ]:
def kucuk_harf(yazi):
    # python'un lower() fonksiyonu türkçe I ve İ harflerinde hata yapıyor
    return yazi.replace("I", "ı").replace("İ", "i").lower()


class TarifDefteri:
    def __init__(self):
        self.tarifler = []
        self.kategoriler = {
            "T": "Tatlı & Kek",
            "C": "Çorba",
            "A": "Ana Yemek",
            "H": "Hamur & Makarna",
            "S": "Salata & Meze",
        }

    def tarif_ekle(self, tarif):
        if tarif.kategori not in self.kategoriler:
            print("Bu kategori yok:", tarif.kategori)
            return
        self.tarifler.append(tarif)

    def ara(self, kelime):
        sonuc = []
        for tarif in self.tarifler:
            if kucuk_harf(kelime) in kucuk_harf(tarif.ad):
                sonuc.append(tarif)
        return sonuc

    def kategoriye_gore(self, kategori):
        sonuc = []
        for tarif in self.tarifler:
            if tarif.kategori == kategori:
                sonuc.append(tarif)
        return sonuc

    def tarif_sayisi(self):
        return len(self.tarifler)

    def json_verisi(self):
        liste = []
        for tarif in self.tarifler:
            liste.append(tarif.sozluge_cevir())
        return json.dumps(liste, ensure_ascii=False)

## 3. Tarifleri ekleme
Önce boş bir defter oluşturup tarifleri tek tek ekliyorum. Sıralama: ad, kişi sayısı, hazırlık, pişirme, kategori, emoji, resim, malzemeler, yapılışı.

In [ ]:
defter = TarifDefteri()

# ---------- TATLILAR VE KEKLER ----------

defter.tarif_ekle(Tarif(
    "Kabaran Kakaolu Sade Kek", "8-10", "15 dk", "45 dk", "T", "🍰",
    "2026/10/06/kabaran-kakaolu-sade-kek.jpg",
    ["4 yumurta", "1,5 su bardağı şeker", "1 su bardağı sıvı yağ", "1 su bardağı süt",
     "4 su bardağı un", "1 yemek kaşığı nişasta", "2 yemek kaşığı sıcak su", "1 paket vanilya",
     "2 paket kabartma tozu", "3 yemek kaşığı kakao", "2 yemek kaşığı süt (kakaolu kısım için)"],
    ["Malzemeleri oda sıcaklığına getirin, yumurta ile şekeri mikserle kabarana kadar çırpın.",
     "Kakao ve 2 kaşık süt hariç diğer malzemeleri ekleyip çırpmaya devam edin.",
     "Karışımın yaklaşık üçte ikisini yağlanmış kek kalıbına dökün.",
     "Kalan karışıma kakao ve sütü ekleyip çırpın, kalıbın üstüne dökün ve çatalla desen verin.",
     "Önceden ısıtılmış 180 derece fırında 40-45 dakika pişirin, soğuyunca kalıptan çıkarın."]
))

defter.tarif_ekle(Tarif(
    "Kabarma Garantili Kakaolu Sade Kek", "6-8", "10 dk", "40 dk", "T", "🍰",
    "2023/07/28/kabarma-garantili-kakaolu-sade-kek-tarifi-videolu-1-320x213.jpg",
    ["3 yumurta", "1 su bardağı şeker", "1 su bardağı süt", "Yarım su bardağı sıvı yağ",
     "2,5 su bardağı un", "1 paket kabartma tozu", "1 paket vanilya", "2 yemek kaşığı kakao"],
    ["Yumurta ve şekeri köpük köpük olana kadar çırpın.",
     "Süt ve sıvı yağı ekleyip karıştırın.",
     "Un, kabartma tozu ve vanilyayı eleyerek ekleyin.",
     "Hamurun yarısını ayırıp içine kakaoyu karıştırın.",
     "Kalıba önce sade, sonra kakaolu hamuru dökün ve 170 derecede 40 dakika pişirin."]
))

defter.tarif_ekle(Tarif(
    "Çok Kabaran Cevizli Tarçınlı Kek", "6-8", "5 dk", "50 dk", "T", "🍰",
    "2018/11/01/cok-kabaran-cevizli-tarcinli-kek-tarifi-videolu-320x213.jpg",
    ["3 yumurta", "1 su bardağı şeker", "1 su bardağı süt", "2/3 su bardağı sıvı yağ",
     "2,5 su bardağı un", "1 su bardağı iri dövülmüş ceviz", "1 yemek kaşığı tarçın",
     "1 paket kabartma tozu", "1 paket vanilya"],
    ["Yumurta ve şekeri iyice çırpın.",
     "Süt ve yağı ekleyip karıştırın.",
     "Un, tarçın, kabartma tozu ve vanilyayı ekleyip pürüzsüz bir hamur elde edin.",
     "Cevizleri biraz unlayıp hamura ekleyin, böylece dibe çökmezler.",
     "Yağlanmış kalıpta 170 derecede yaklaşık 50 dakika pişirin."]
))

defter.tarif_ekle(Tarif(
    "Starbucks Tadında Havuçlu Kek", "8-10", "25 dk", "45 dk", "T", "🧁",
    "2019/08/19/kabardikca-kabaran-starbucks-keki-videolu-320x213.jpg",
    ["3 yumurta", "1 su bardağı esmer şeker", "3/4 su bardağı sıvı yağ", "2 orta boy havuç (rendelenmiş)",
     "2 su bardağı un", "1 çay kaşığı tarçın", "1 paket kabartma tozu", "Yarım su bardağı ceviz"],
    ["Havuçları ince rendeleyin.",
     "Yumurta ve esmer şekeri çırpın, sıvı yağı ekleyin.",
     "Un, tarçın ve kabartma tozunu ekleyip karıştırın.",
     "Havuç ve cevizi ekleyip spatula ile karıştırın.",
     "180 derece fırında 45 dakika pişirin, kürdan temiz çıkınca alın."]
))

defter.tarif_ekle(Tarif(
    "Yumuşacık Kakaolu Muffin Kek", "12-14", "10 dk", "20 dk", "T", "🧁",
    "2020/10/07/yumusacik-kakaolu-muffin-kek-tarifi-videolu-320x213.jpg",
    ["2 yumurta", "1 su bardağı şeker", "1 su bardağı süt", "Yarım su bardağı sıvı yağ",
     "2 su bardağı un", "3 yemek kaşığı kakao", "1 paket kabartma tozu", "Yarım su bardağı damla çikolata"],
    ["Yumurta ve şekeri çırpın.",
     "Süt ve sıvı yağı ekleyin.",
     "Un, kakao ve kabartma tozunu eleyerek ekleyin.",
     "Damla çikolataları ekleyip karıştırın.",
     "Muffin kalıplarını 3'te 2'sine kadar doldurup 180 derecede 20 dakika pişirin."]
))

defter.tarif_ekle(Tarif(
    "Kakaolu Islak Kek", "6-8", "15 dk", "45 dk", "T", "🍫",
    "2023/04/26/kakaolu-islak-kek-8-320x213.jpg",
    ["3 yumurta", "1 su bardağı şeker", "1 su bardağı süt", "3/4 su bardağı sıvı yağ",
     "4 yemek kaşığı kakao", "1,5 su bardağı un", "1 paket kabartma tozu", "1 paket vanilya",
     "Sos için: 1,5 su bardağı süt, 2 yemek kaşığı kakao, 2 yemek kaşığı şeker, 1 yemek kaşığı tereyağı"],
    ["Yumurta ve şekeri çırpın, süt ve yağı ekleyin.",
     "Kakao, un, kabartma tozu ve vanilyayı ekleyip karıştırın.",
     "Hamuru kalıba döküp 175 derecede 35-40 dakika pişirin.",
     "Sos malzemelerini tencerede kaynayana kadar karıştırın.",
     "Kek sıcakken çatalla delip sosun çoğunu üstüne dökün, buzdolabında 1-2 saat dinlendirin."]
))

defter.tarif_ekle(Tarif(
    "Portakallı Kakaolu Kek", "8-10", "5 dk", "50 dk", "T", "🍊",
    "2022/12/15/portakalli-kakaolu-kek-12-320x213.jpg",
    ["3 yumurta", "1 su bardağı şeker", "1 portakalın suyu ve kabuk rendesi", "Yarım su bardağı sıvı yağ",
     "2,5 su bardağı un", "2 yemek kaşığı kakao", "1 paket kabartma tozu", "1 paket vanilya"],
    ["Yumurta ve şekeri çırpın.",
     "Portakal suyu, kabuk rendesi ve sıvı yağı ekleyin.",
     "Un, kabartma tozu ve vanilyayı ekleyip karıştırın.",
     "Hamurun yarısına kakao ekleyin ve iki hamuru kalıba sırayla dökün.",
     "170 derecede 50 dakika pişirin."]
))

defter.tarif_ekle(Tarif(
    "Çok Kabaran Sulu Kek", "6-8", "15 dk", "40 dk", "T", "🍰",
    "2018/10/15/cok-kabaran-sulu-kek-yapimi-videolu-320x213.jpg",
    ["3 yumurta", "1 su bardağı şeker", "1 su bardağı süt", "Yarım su bardağı sıvı yağ",
     "2 su bardağı un", "3 yemek kaşığı kakao", "1 paket kabartma tozu",
     "Sos için: 2 su bardağı süt, 1 yemek kaşığı kakao, 2 yemek kaşığı şeker"],
    ["Kek malzemelerini sırayla çırparak karıştırın.",
     "Yağlanmış tepsiye döküp 180 derecede 35-40 dakika pişirin.",
     "Sos malzemelerini ılık olacak şekilde ısıtın.",
     "Fırından çıkan kekin üstüne ılık sosu yavaş yavaş dökün.",
     "Sosu çekince dilimleyip servis edin."]
))

defter.tarif_ekle(Tarif(
    "Vişneli Sade Kek", "12-14", "25 dk", "40 dk", "T", "🍒",
    "2017/09/13/visneli-sade-kek-yapimi-resimli-ve-videolu-320x213.jpg",
    ["3 yumurta", "1 su bardağı şeker", "1 su bardağı yoğurt", "Yarım su bardağı sıvı yağ",
     "2,5 su bardağı un", "1 su bardağı çekirdeği çıkarılmış vişne", "1 paket kabartma tozu", "1 paket vanilya"],
    ["Vişnelerin çekirdeklerini çıkarıp suyunu süzün.",
     "Yumurta ve şekeri çırpın, yoğurt ve yağı ekleyin.",
     "Un, kabartma tozu ve vanilyayı ekleyin.",
     "Vişneleri bir kaşık unla karıştırıp hamura ekleyin.",
     "175 derecede 40 dakika pişirin."]
))

defter.tarif_ekle(Tarif(
    "Ispanaklı Kakaolu Kek", "12-14", "10 dk", "40 dk", "T", "🍫",
    "2017/02/17/ispanakli-kakaolu-kek-videosu-320x213.jpg",
    ["Yarım bağ ıspanak", "3 yumurta", "1 su bardağı şeker", "3/4 su bardağı sıvı yağ",
     "2,5 su bardağı un", "2 yemek kaşığı kakao", "1 paket kabartma tozu", "1 paket vanilya"],
    ["Ispanakları yıkayıp 2 dakika haşlayın ve blenderdan geçirin.",
     "Yumurta ve şekeri çırpın, yağ ve ıspanak püresini ekleyin.",
     "Un, kabartma tozu ve vanilyayı ekleyip karıştırın.",
     "Hamurun yarısına kakao ekleyin, iki hamuru kalıba sırayla dökün.",
     "175 derecede 40 dakika pişirin."]
))

defter.tarif_ekle(Tarif(
    "Ezber Bozan Kek", "10-12", "15 dk", "40 dk", "T", "🍰",
    "2016/09/22/ezber-bozan-kek-kabaran-bayatlamayan-videosu-320x180.jpg",
    ["4 yumurta (sarısı ve beyazı ayrı)", "1 su bardağı şeker", "1 su bardağı süt", "1 su bardağı sıvı yağ",
     "2,5 su bardağı un", "1 yemek kaşığı nişasta", "1 paket kabartma tozu", "1 paket vanilya"],
    ["Yumurta beyazlarını bir tutam tuzla kar gibi olana kadar çırpın.",
     "Başka bir kapta sarıları şekerle çırpın, süt ve yağı ekleyin.",
     "Un, nişasta, kabartma tozu ve vanilyayı ekleyip karıştırın.",
     "Çırpılmış beyazları spatula ile yavaşça hamura katın.",
     "170 derecede 40 dakika pişirin, ilk 30 dakika fırın kapağını açmayın."]
))

defter.tarif_ekle(Tarif(
    "Süt Helvası", "4-6", "5 dk", "20 dk", "T", "🍮",
    "2026/10/05/sut-helvasi-600x400.jpg",
    ["1 litre süt", "1 su bardağı un", "125 gr tereyağı", "1,5 su bardağı şeker",
     "1 paket vanilya", "2 yemek kaşığı dolmalık fıstık"],
    ["Sütü şekerle birlikte ayrı bir tencerede ısıtın.",
     "Tereyağını eritip fıstıkları hafifçe kavurun.",
     "Unu ekleyip rengi çok koyulaşmadan 5-6 dakika kavurun.",
     "Sıcak sütü azar azar ekleyip sürekli karıştırın.",
     "Koyulaşınca vanilyayı ekleyin, kapağını kapatıp 10 dakika demlendirin."]
))

defter.tarif_ekle(Tarif(
    "Şeftalili Crumble Cheesecake", "6-8", "30 dk", "1 saat", "T", "🍑",
    "2026/10/02/seftalili-crumble-cheesecake-320x213.jpg",
    ["1 paket bisküvi", "75 gr eritilmiş tereyağı", "400 gr labne", "200 ml krema",
     "3/4 su bardağı şeker", "2 yumurta", "1 yemek kaşığı nişasta", "2 şeftali",
     "Crumble için: yarım su bardağı un, 3 yemek kaşığı şeker, 50 gr soğuk tereyağı"],
    ["Bisküvileri ufalayıp tereyağı ile karıştırın ve kalıbın tabanına bastırın.",
     "Labne, krema, şeker, yumurta ve nişastayı pürüzsüz olana kadar karıştırın.",
     "Kremayı tabanın üstüne dökün, küp doğranmış şeftalileri üstüne dizin.",
     "Crumble malzemelerini parmak uçlarıyla ufalayıp en üste serpin.",
     "160 derecede 1 saat pişirin, soğuyunca bir gece buzdolabında bekletin."]
))

defter.tarif_ekle(Tarif(
    "Kakaolu Muzlu Krep", "2-4", "10 dk", "15 dk", "T", "🥞",
    "2026/10/06/cocuklarin-favorisi-kakaolu-muzlu-krep-320x213.jpg",
    ["2 yumurta", "1 su bardağı süt", "1 su bardağı un", "1 yemek kaşığı kakao",
     "1 yemek kaşığı şeker", "1 yemek kaşığı sıvı yağ", "1 muz", "Fındık kreması"],
    ["Yumurta, süt, un, kakao, şeker ve yağı topaksız olana kadar çırpın.",
     "Hamuru 10 dakika dinlendirin.",
     "Yağlanmış tavaya bir kepçe hamur döküp ince yayın, iki tarafını pişirin.",
     "Krepin üstüne fındık kreması sürüp muz dilimlerini dizin.",
     "Rulo veya üçgen şeklinde katlayıp servis edin."]
))

defter.tarif_ekle(Tarif(
    "Kahve Soslu Browni Kurabiye", "10-12", "15 dk", "20 dk", "T", "🍪",
    "2026/10/06/kahve-soslu-browni-kurabiye-320x213.jpg",
    ["100 gr bitter çikolata", "50 gr tereyağı", "1 yumurta", "Yarım su bardağı şeker",
     "1 su bardağı un", "1 yemek kaşığı kakao", "Yarım çay kaşığı kabartma tozu",
     "Sos için: 1 çay kaşığı granül kahve, 2 yemek kaşığı sıcak su, 3 yemek kaşığı pudra şekeri"],
    ["Çikolata ve tereyağını benmari usulü eritin.",
     "Yumurta ve şekeri çırpıp çikolatalı karışıma ekleyin.",
     "Un, kakao ve kabartma tozunu ekleyip koyu bir hamur yapın.",
     "Ceviz büyüklüğünde parçalar alıp tepsiye dizin, 175 derecede 10-12 dakika pişirin.",
     "Kahveyi sıcak suda eritip pudra şekeriyle karıştırın ve soğuyan kurabiyelerin üstüne gezdirin."]
))

print("Tatlılar eklendi:", defter.tarif_sayisi())

In [ ]:
# ---------- ÇORBALAR ----------

defter.tarif_ekle(Tarif(
    "Tavuklu Tatlı Patates Çorbası", "4-6", "5 dk", "30 dk", "C", "🥣",
    "2026/10/05/tavuklu-tatli-patates-corbasi-1-320x213.jpg",
    ["1 adet tavuk göğsü", "2 tatlı patates", "1 soğan", "1 havuç", "1 yemek kaşığı tereyağı",
     "5 su bardağı tavuk suyu", "Tuz, karabiber"],
    ["Tavuk göğsünü haşlayıp didikleyin, suyunu ayırın.",
     "Soğan ve havucu tereyağında kavurun.",
     "Küp doğranmış tatlı patatesi ve tavuk suyunu ekleyip yumuşayana kadar pişirin.",
     "Blenderdan geçirin, tavukları ekleyip birkaç dakika daha kaynatın.",
     "Tuz ve karabiberini ayarlayıp servis edin."]
))

defter.tarif_ekle(Tarif(
    "Kars Yöresine Ait Kesme Çorbası", "4-6", "30 dk", "25 dk", "C", "🥣",
    "2026/10/02/kars-yoresine-ait-kesme-corbasi-1-320x213.jpg",
    ["Hamur için: 1 yumurta, 1 su bardağı un, yarım çay kaşığı tuz, biraz su",
     "1 su bardağı haşlanmış yeşil mercimek", "1 soğan", "1 yemek kaşığı salça",
     "2 yemek kaşığı tereyağı", "6 su bardağı su", "1 tatlı kaşığı kuru nane", "Tuz"],
    ["Yumurta, un, tuz ve azar azar su ile sert bir hamur yoğurun.",
     "Hamuru ince açıp biraz kurutun, sonra küçük kareler halinde kesin.",
     "Soğanı tereyağında kavurup salçayı ekleyin.",
     "Su ve mercimeği ekleyip kaynatın, kesme hamurları ilave edin.",
     "Hamurlar pişince nane ve tuz ekleyip ocaktan alın."]
))

# ---------- ANA YEMEKLER ----------

defter.tarif_ekle(Tarif(
    "Tavuk Kalamar", "4-6", "10 dk", "20 dk", "A", "🍗",
    "2026/10/05/tavuk-kalamar-600x400.jpg",
    ["500 gr tavuk göğsü", "1 su bardağı un", "2 yumurta", "1 su bardağı galeta unu",
     "Tuz, karabiber, toz kırmızı biber", "Kızartmak için sıvı yağ"],
    ["Tavuk göğsünü ince şeritler halinde kesin, baharatlarla harmanlayın.",
     "Unu, çırpılmış yumurtayı ve galeta ununu ayrı kaplara koyun.",
     "Tavukları sırayla una, yumurtaya ve galeta ununa bulayın.",
     "Kızgın yağda altın rengi olana kadar kızartın.",
     "Kağıt havlu üzerine alıp yağını süzdürün, sıcak servis edin."]
))

defter.tarif_ekle(Tarif(
    "Kıyma (Tantuni Tarzı)", "2-4", "10 dk", "35 dk", "A", "🥩",
    "2026/10/06/kiyma-tantuni-tarzi-320x213.jpg",
    ["300 gr kıyma", "1 soğan", "2 domates", "1 yemek kaşığı sıvı yağ",
     "Pul biber, kimyon, tuz", "Yarım demet maydanoz", "Lavaş"],
    ["Kıymayı az yağda suyunu çekene kadar kavurun.",
     "Baharatları ekleyip birkaç dakika daha kavurun.",
     "Soğanı piyazlık doğrayıp maydanozla karıştırın, domatesleri küp doğrayın.",
     "Lavaşı tavada ısıtıp kıymayı, soğanı ve domatesi içine koyun.",
     "Dürüm yapıp sıcak servis edin."]
))

defter.tarif_ekle(Tarif(
    "Harran Kubbesi", "2-4", "5 dk", "1 saat", "A", "🥘",
    "2026/10/05/harran-kubbesi-320x213.jpg",
    ["2 su bardağı ince bulgur", "250 gr yağsız kıyma (dış harç için)", "200 gr kıyma (iç harç için)",
     "1 soğan", "Yarım su bardağı ceviz", "1 yemek kaşığı biber salçası",
     "Pul biber, kimyon, tuz", "2 yemek kaşığı tereyağı"],
    ["Bulguru az suyla ıslatıp yağsız kıyma, salça ve baharatla iyice yoğurun.",
     "İç harç için kıymayı soğanla kavurup ceviz ve baharatları ekleyin.",
     "Dış harcın yarısını yağlanmış tepsiye yayın, iç harcı üstüne serin.",
     "Kalan dış harçla üstünü kapatın, dilimleyip tereyağı gezdirin.",
     "200 derece fırında yaklaşık 1 saat pişirin."]
))

defter.tarif_ekle(Tarif(
    "İspir Fasulyesi", "4-6", "10 dk", "30 dk", "A", "🫘",
    "2026/10/06/ispir-fasulyesi-320x213.jpg",
    ["2 su bardağı İspir kuru fasulyesi (bir gece önceden ıslatılmış)", "1 soğan",
     "1 yemek kaşığı domates salçası", "2 yemek kaşığı tereyağı", "4 su bardağı sıcak su",
     "Tuz, pul biber"],
    ["Islatılmış fasulyeleri süzün.",
     "Soğanı tereyağında kavurup salçayı ekleyin.",
     "Fasulyeleri ve sıcak suyu ekleyin.",
     "Düdüklü tencerede 25-30 dakika pişirin.",
     "Tuzunu ekleyip birkaç dakika daha kaynatın, pilavla servis edin."]
))

defter.tarif_ekle(Tarif(
    "Katıklı Kabak", "4-6", "15 dk", "30 dk", "A", "🎃",
    "2026/10/06/katikli-kabak-320x213.jpg",
    ["3 kabak", "1 soğan", "2 yemek kaşığı zeytinyağı", "1 su bardağı yoğurt",
     "2 diş sarımsak", "1 yemek kaşığı tereyağı", "Pul biber, tuz"],
    ["Kabakları yarım ay şeklinde doğrayın.",
     "Soğanı zeytinyağında kavurun, kabakları ekleyip yumuşayana kadar pişirin.",
     "Yoğurdu ezilmiş sarımsak ve tuzla çırpın.",
     "Kabakları tabağa alıp üstüne sarımsaklı yoğurdu (katık) dökün.",
     "Tereyağını pul biberle kızdırıp üstüne gezdirin."]
))

# ---------- HAMUR İŞLERİ VE MAKARNALAR ----------

defter.tarif_ekle(Tarif(
    "Çiğ Börek", "4-6", "20 dk", "15 dk", "H", "🥟",
    "2026/10/05/cig-borek-5-320x213.jpg",
    ["3 su bardağı un", "1 su bardağı ılık su", "1 çay kaşığı tuz", "250 gr kıyma",
     "1 soğan (rendelenmiş)", "Tuz, karabiber", "Kızartmak için sıvı yağ"],
    ["Un, su ve tuzla yumuşak bir hamur yoğurup 20 dakika dinlendirin.",
     "Kıyma, soğan rendesi ve baharatları karıştırıp iç harcı hazırlayın.",
     "Hamuru bezelere ayırıp ince açın, yarısına iç harcı yayın.",
     "Hamuru kapatıp kenarlarını bastırın ve kesin.",
     "Kızgın yağda iki tarafını da kızartın."]
))

defter.tarif_ekle(Tarif(
    "Mardin Çöreği (Kliçe)", "8-10", "20 dk", "15 dk", "H", "🥯",
    "2026/10/04/mardin-coregi-klice-320x213.jpg",
    ["1 su bardağı ılık süt", "Yarım su bardağı sıvı yağ", "125 gr eritilmiş tereyağı",
     "1 paket instant maya", "2 yemek kaşığı şeker", "1 çay kaşığı tuz",
     "1 yemek kaşığı çörek baharatı (mahlep, rezene, çörek otu)", "4 su bardağı un",
     "Üzeri için 1 yumurta sarısı"],
    ["Süt, yağlar, maya, şeker ve tuzu karıştırın.",
     "Baharatı ve unu azar azar ekleyerek yumuşak bir hamur yoğurun.",
     "Hamuru 30 dakika mayalandırın.",
     "Bezelere ayırıp yuvarlak açın, kenarlarına desen verin ve üstüne yumurta sarısı sürün.",
     "180 derecede üstü kızarana kadar yaklaşık 15 dakika pişirin."]
))

defter.tarif_ekle(Tarif(
    "Domatesli Zeytinli Focaccia", "6-8", "3 saat", "20 dk", "H", "🍞",
    "2026/10/06/domatesli-zeytinli-focaccia-320x213.jpg",
    ["3,5 su bardağı un", "1,5 su bardağı ılık su", "1 paket instant maya", "1 çay kaşığı şeker",
     "1 çay kaşığı tuz", "5 yemek kaşığı zeytinyağı", "10 adet çeri domates",
     "10 adet siyah zeytin", "Kekik, iri tuz"],
    ["Un, su, maya, şeker ve tuzu karıştırıp cıvık bir hamur yapın.",
     "Üstüne 2 kaşık zeytinyağı gezdirip kapağı kapalı şekilde 2 saat mayalandırın.",
     "Yağlanmış tepsiye alıp parmaklarla yayın ve 1 saat daha dinlendirin.",
     "Parmak uçlarıyla çukurlar açın, domates ve zeytinleri bastırın, kalan zeytinyağı, kekik ve iri tuzu serpin.",
     "220 derecede 20 dakika pişirin."]
))

defter.tarif_ekle(Tarif(
    "Bolonez Soslu Makarna", "4-6", "20 dk", "15 dk", "H", "🍝",
    "2026/10/06/bolonez-soslu-makarna-320x213.jpg",
    ["1 paket spagetti", "250 gr kıyma", "1 soğan", "1 havuç", "2 diş sarımsak",
     "2 domates (rendelenmiş)", "1 yemek kaşığı domates salçası", "2 yemek kaşığı zeytinyağı",
     "Tuz, karabiber", "Üzeri için rendelenmiş parmesan"],
    ["Soğan, havuç ve sarımsağı ince doğrayıp zeytinyağında kavurun.",
     "Kıymayı ekleyip suyunu çekene kadar pişirin.",
     "Salça ve domatesi ekleyip sosu 10 dakika kısık ateşte pişirin.",
     "Makarnayı tuzlu suda haşlayıp süzün.",
     "Makarnayı sosla karıştırıp üstüne parmesan serpin."]
))

defter.tarif_ekle(Tarif(
    "Krema Lezzetinde Sütlü Erişte", "4-6", "10 dk", "20 dk", "H", "🍜",
    "2026/10/05/krema-lezzetinde-sutlu-eriste-600x400.jpg",
    ["2 su bardağı erişte", "3 su bardağı süt", "1 su bardağı su", "1 yemek kaşığı tereyağı",
     "Yarım su bardağı rendelenmiş kaşar", "Tuz, karabiber"],
    ["Erişteyi tereyağında 2-3 dakika kavurun.",
     "Süt ve suyu ekleyip kaynamaya bırakın.",
     "Kısık ateşte erişteler sütü çekene kadar karıştırarak pişirin.",
     "Tuz ve karabiberi ekleyin.",
     "Üstüne kaşar serpip kapağı kapalı şekilde eriyene kadar bekletin."]
))

# ---------- SALATALAR VE MEZELER ----------

defter.tarif_ekle(Tarif(
    "Humus", "2-4", "10 dk", "Pişirme yok", "S", "🧆",
    "2026/10/05/humus-efsane-tarif-320x213.jpg",
    ["2 su bardağı haşlanmış nohut", "3 yemek kaşığı tahin", "1 limonun suyu", "1 diş sarımsak",
     "3 yemek kaşığı zeytinyağı", "3-4 yemek kaşığı buzlu su", "Kimyon, tuz", "Üzeri için pul biber"],
    ["Nohutların kabuklarını mümkün olduğunca soyun.",
     "Nohut, tahin, limon suyu, sarımsak, kimyon ve tuzu robottan geçirin.",
     "Buzlu suyu azar azar ekleyerek kremamsı bir kıvam yakalayın.",
     "Servis tabağına alıp kaşıkla ortasını çukurlaştırın.",
     "Üstüne zeytinyağı gezdirip pul biber serpin."]
))

defter.tarif_ekle(Tarif(
    "Çıtır Marul Salatası", "1-2", "5 dk", "5 dk", "S", "🥗",
    "2026/10/05/citir-marul-salatasi-320x213.jpg",
    ["Yarım marul", "1 salatalık", "2 dal yeşil soğan", "Birkaç dal dereotu", "1 dilim bayat ekmek",
     "1 yemek kaşığı zeytinyağı", "Yarım limonun suyu", "1 çay kaşığı nar ekşisi", "Tuz"],
    ["Ekmeği küp doğrayıp tavada çıtır olana kadar kızartın.",
     "Marulu yıkayıp ince ince doğrayın.",
     "Salatalık, yeşil soğan ve dereotunu doğrayıp marula ekleyin.",
     "Zeytinyağı, limon, nar ekşisi ve tuzla sos hazırlayıp salataya dökün.",
     "Kıtır ekmekleri üstüne serpip hemen servis edin."]
))

defter.tarif_ekle(Tarif(
    "Şifalı Turp Turşusu", "6-8", "15 dk", "Pişirme yok", "S", "🥒",
    "2026/10/01/sahane-sifali-turp-tursusu-yap-ye-320x213.jpg",
    ["1 kg turp", "4 diş sarımsak", "1 dilim pancar (renk için)", "1 su bardağı elma sirkesi",
     "2,5 su bardağı su", "1 yemek kaşığı kaya tuzu", "1 çay kaşığı şeker"],
    ["Turpları yıkayıp kabuklarıyla birlikte dilimleyin.",
     "Turpları, sarımsakları ve pancarı cam kavanoza yerleştirin.",
     "Su, sirke, tuz ve şekeri karıştırıp kavanoza dökün.",
     "Kavanozun ağzını sıkıca kapatın.",
     "Serin bir yerde 10-15 gün bekletip öyle tüketin."]
))

print("Toplam tarif sayısı:", defter.tarif_sayisi())

### Python tarafında küçük bir kontrol
Arayüzü yapmadan önce sınıfların doğru çalışıp çalışmadığına baktım.

In [ ]:
print("İçinde 'kek' geçen tarifler:")
for tarif in defter.ara("kek"):
    print("  -", tarif)

print()
for kod, isim in defter.kategoriler.items():
    print(isim, "->", len(defter.kategoriye_gore(kod)), "tarif")

## 4. Saklama süreleri
Her yemeğin buzdolabında (4 °C ve altı, kapalı kapta) ortalama kaç gün dayandığını araştırıp ekledim.

**Araştırırken kullandığım genel kurallar:**
- ABD Tarım Bakanlığı (USDA FSIS): pişmiş yemek artıkları buzdolabında **3-4 gün** dayanır, dondurucuda 3-4 ay saklanabilir.
- Pişen yemek en geç **2 saat** içinde dolaba kaldırılmalı, 2 saatten uzun oda sıcaklığında kalan yemek atılmalı.
- Çorbalar 3-4 gün, soslu makarna 3-4 gün (sade makarna 3-5 gün), pişmiş kuru baklagiller 3-5 gün.
- Cheesecake dolapta 5-7 gün; kremasız ev keki oda sıcaklığında 2-3 gün, dolapta yaklaşık 1 hafta.
- Sütlü / yoğurtlu yemekler ve soslu salatalar daha kısa dayanır.

Kaynaklar: [USDA FSIS – Leftovers and Food Safety](https://www.fsis.usda.gov/food-safety/safe-food-handling-and-preparation/food-safety-basics/leftovers-and-food-safety), [USDA FoodKeeper](https://ifdaonline.org/wp-content/uploads/2024/02/food-keeper-2015-pdf.pdf)

> Bunlar ortalama sürelerdir. Kokusu, görüntüsü veya tadı değişen yemek süresi dolmamış olsa bile tüketilmemeli.

Sıralama: tarif adı → (en az gün, en çok gün, açıklama)

In [ ]:
SADE_KEK = "Kremasız kekler kapalı kapta oda sıcaklığında 2-3 gün, dolapta yaklaşık 1 hafta dayanır. Kurumaması için streç filme sarın."
SUTLU_SOS = "Süt bazlı sos içerdiği için mutlaka buzdolabında saklanmalı."

SAKLAMA_SURELERI = {
    # tatlılar ve kekler
    "Kabaran Kakaolu Sade Kek": (5, 7, SADE_KEK),
    "Kabarma Garantili Kakaolu Sade Kek": (5, 7, SADE_KEK),
    "Çok Kabaran Cevizli Tarçınlı Kek": (5, 7, SADE_KEK),
    "Starbucks Tadında Havuçlu Kek": (5, 7, "Havuç kekin nemli kalmasını sağlar; oda sıcaklığında 2 gün, dolapta 1 haftaya kadar dayanır."),
    "Yumuşacık Kakaolu Muffin Kek": (5, 7, SADE_KEK),
    "Kakaolu Islak Kek": (3, 4, SUTLU_SOS),
    "Portakallı Kakaolu Kek": (5, 7, SADE_KEK),
    "Çok Kabaran Sulu Kek": (3, 4, SUTLU_SOS),
    "Vişneli Sade Kek": (4, 5, "Meyveli ve yoğurtlu olduğu için çabuk nemlenir; dolapta kapalı kapta saklayın."),
    "Ispanaklı Kakaolu Kek": (4, 5, "Sebze püresi içerdiği için sade keklere göre biraz daha kısa dayanır."),
    "Ezber Bozan Kek": (5, 7, SADE_KEK),
    "Süt Helvası": (3, 4, "Sütlü tatlılar gibi buzdolabında saklanmalı; ısıtarak tüketilebilir."),
    "Şeftalili Crumble Cheesecake": (5, 7, "Cheesecake dolapta 5-7 gün dayanır. Taze meyveli olduğu için ilk günlerde yemek daha iyi."),
    "Kakaolu Muzlu Krep": (1, 2, "Muz kısa sürede kararır, muzlu krep hemen yenmeli. Sade krepler dolapta 3-5 gün saklanabilir."),
    "Kahve Soslu Browni Kurabiye": (5, 7, "Kapalı kapta oda sıcaklığında 4-5 gün, dolapta 1 haftaya kadar dayanır."),
    # çorbalar
    "Tavuklu Tatlı Patates Çorbası": (3, 4, "Çorbalar dolapta 3-4 gün dayanır. Her seferinde sadece içeceğiniz kadarını ısıtın."),
    "Kars Yöresine Ait Kesme Çorbası": (3, 4, "Kesme hamurlar bekledikçe suyu çeker; ısıtırken biraz su ekleyin."),
    # ana yemekler
    "Tavuk Kalamar": (3, 4, "Pişmiş tavuk dolapta 3-4 gün dayanır; fırında ısıtırsanız çıtırlığı geri gelir."),
    "Kıyma (Tantuni Tarzı)": (3, 4, "Kıymalı harcı ayrı saklayın; lavaşa sarılmış hali aynı gün yenmeli."),
    "Harran Kubbesi": (3, 4, "Pişmiş etli yemekler dolapta 3-4 gün dayanır, dondurucuda 2-3 ay saklanabilir."),
    "İspir Fasulyesi": (3, 5, "Pişmiş kuru baklagil yemekleri dolapta 3-5 gün dayanır."),
    "Katıklı Kabak": (2, 3, "Sarımsaklı yoğurtlu hali 2-3 gün içinde yenmeli; mümkünse yoğurdu ayrı saklayın."),
    # hamur işleri ve makarnalar
    "Çiğ Börek": (2, 3, "Kızartılmış etli hamur işleri dolapta 2-3 gün dayanır ama en lezzetlisi ilk gün."),
    "Mardin Çöreği (Kliçe)": (5, 7, "Kapalı poşette oda sıcaklığında 3-4 gün, dolapta 1 haftaya kadar dayanır. Uzun süre için dondurun."),
    "Domatesli Zeytinli Focaccia": (3, 4, "Taze domatesli olduğu için dolapta saklanmalı; yemeden önce fırında ısıtın."),
    "Bolonez Soslu Makarna": (3, 4, "Soslu makarna dolapta 3-4 gün dayanır. Sos ayrı saklanırsa sade makarna 3-5 gün dayanır."),
    "Krema Lezzetinde Sütlü Erişte": (2, 3, "Sütlü ve kaşarlı olduğu için kısa sürede tüketilmeli; ısıtırken biraz süt ekleyin."),
    # salatalar ve mezeler
    "Humus": (3, 5, "Ev yapımı humus hava almayan kapta, üstüne biraz zeytinyağı gezdirilerek 3-5 gün saklanır."),
    "Çıtır Marul Salatası": (1, 1, "Soslu yeşil salata aynı gün yenmeli. Sossuz ve ekmeksiz hali dolapta 2-3 gün dayanır."),
    "Şifalı Turp Turşusu": (60, 90, "Turşu kurulduktan 10-15 gün sonra açılır; açıldıktan sonra dolapta 2-3 ay saklanabilir."),
}

for tarif in defter.tarifler:
    if tarif.ad in SAKLAMA_SURELERI:
        en_az, en_cok, aciklama = SAKLAMA_SURELERI[tarif.ad]
        tarif.saklama_ekle(en_az, en_cok, aciklama)
    else:
        print("Saklama bilgisi eksik:", tarif.ad)

for tarif in defter.tarifler[:5]:
    print(f"{tarif.ad:40} 🧊 {tarif.saklama_yazisi()}")
print("...")

## 5. Buzdolabi sınıfı
Tariflerde geçen temel malzemeleri gruplara ayırdım. Her tarifin malzeme yazılarında bu malzemelerden hangilerinin geçtiğini buluyorum (örn. *"2,5 su bardağı un"* → `un`).
Sonra dolapta olan malzemelerle karşılaştırıp her tarif için **uyum yüzdesi** ve **eksik malzemeleri** hesaplıyorum.

Tuz, karabiber, su ve baharatları her evde olduğu için hesaba katmadım.

In [ ]:
MALZEME_GRUPLARI = {
    "Süt ürünleri & Yumurta": ["yumurta", "süt", "yoğurt", "tereyağı", "krema", "labne", "kaşar", "parmesan"],
    "Et & Tavuk": ["tavuk", "kıyma"],
    "Sebze": ["soğan", "sarımsak", "domates", "havuç", "patates", "kabak", "ıspanak", "marul",
              "salatalık", "maydanoz", "dereotu", "turp", "pancar"],
    "Meyve": ["portakal", "limon", "muz", "şeftali", "vişne"],
    "Kuru gıda & Bakliyat": ["un", "şeker", "nişasta", "kabartma tozu", "vanilya", "maya", "kakao",
                             "çikolata", "ceviz", "fıstık", "nohut", "mercimek", "bulgur", "fasulye",
                             "makarna", "erişte", "galeta unu", "bisküvi"],
    "Yağ, Sos & Diğer": ["sıvı yağ", "zeytinyağı", "salça", "tahin", "zeytin", "sirke", "nar ekşisi",
                         "ekmek", "lavaş", "tarçın", "kahve", "fındık kreması"],
}

# bazı malzemeler tariflerde farklı yazılmış (limonun suyu, domates salçası gibi)
EK_YAZILISLAR = {
    "portakal": ["portakalın"],
    "limon": ["limonun"],
    "salça": ["salçası"],
    "fasulye": ["fasulyesi"],
    "makarna": ["spagetti"],
    "sirke": ["sirkesi"],
    "şeker": ["şekeri"],
}


def tum_malzemeler():
    liste = []
    for grup in MALZEME_GRUPLARI.values():
        liste += grup
    return liste


class Buzdolabi:
    def __init__(self, defter):
        self.defter = defter
        self.dolaptakiler = []
        self.tarifleri_tara()

    def tarifte_var_mi(self, malzeme, tarif):
        metin = kucuk_harf(" ".join(tarif.malzemeler))
        for kelime in [malzeme] + EK_YAZILISLAR.get(malzeme, []):
            # kelimenin tek başına geçmesi lazım: "un" ararken "galeta unu" sayılmasın
            if re.search(r"(?<!\w)" + re.escape(kelime) + r"(?!\w)", metin):
                return True
        return False

    def tarifleri_tara(self):
        # her tarifin hangi temel malzemeleri kullandığını bulup tarife kaydediyorum
        for tarif in self.defter.tarifler:
            tarif.anahtar_malzemeler = []
            for malzeme in tum_malzemeler():
                if self.tarifte_var_mi(malzeme, tarif):
                    tarif.anahtar_malzemeler.append(malzeme)

    def ekle(self, *malzemeler):
        for malzeme in malzemeler:
            malzeme = kucuk_harf(malzeme)
            if malzeme not in tum_malzemeler():
                print("Tariflerde böyle bir malzeme yok:", malzeme)
            elif malzeme not in self.dolaptakiler:
                self.dolaptakiler.append(malzeme)

    def cikar(self, malzeme):
        if malzeme in self.dolaptakiler:
            self.dolaptakiler.remove(malzeme)

    def ne_pisirebilirim(self, en_az_uyum=0):
        sonuc = []
        for tarif in self.defter.tarifler:
            olanlar = [m for m in tarif.anahtar_malzemeler if m in self.dolaptakiler]
            eksikler = [m for m in tarif.anahtar_malzemeler if m not in self.dolaptakiler]
            if len(olanlar) == 0:
                continue
            uyum = len(olanlar) / len(tarif.anahtar_malzemeler)
            if uyum >= en_az_uyum:
                sonuc.append((tarif, uyum, eksikler))
        # önce uyumu yüksek olan, eşitse eksiği az olan gelsin
        sonuc.sort(key=lambda x: (-x[1], len(x[2])))
        return sonuc


buzdolabi = Buzdolabi(defter)
print("Bolonez Soslu Makarna ->", defter.ara("bolonez")[0].anahtar_malzemeler)
print("Humus ->", defter.ara("humus")[0].anahtar_malzemeler)

In [ ]:
# deneme: dolapta bunlar olsun
buzdolabi.ekle("yumurta", "süt", "un", "şeker", "sıvı yağ", "kakao", "kabartma tozu", "soğan", "tereyağı")

print("Dolaptakiler:", ", ".join(buzdolabi.dolaptakiler))
print()
for tarif, uyum, eksikler in buzdolabi.ne_pisirebilirim(en_az_uyum=0.6):
    if eksikler:
        eksik_yazi = "eksik: " + ", ".join(eksikler)
    else:
        eksik_yazi = "hepsi var ✅"
    print(f"%{round(uyum * 100):3}  {tarif.ad:38} {eksik_yazi}")

## 6. YemekGunlugu sınıfı
Her gün hangi öğünde ne yediğimizi kaydediyor ve pazartesiden pazara haftalık liste çıkarıyor.
Yenen yemek defterdeki bir tarifse, saklama süresine bakarak **artanın en geç ne zamana kadar yenmesi gerektiğini** de hesaplıyor.
Defterde olmayan yemekler için genel kural olan 3 günü kullanıyorum.

In [ ]:
OGUNLER = ["Kahvaltı", "Öğle", "Akşam", "Ara öğün"]
GUNLER = ["Pazartesi", "Salı", "Çarşamba", "Perşembe", "Cuma", "Cumartesi", "Pazar"]
VARSAYILAN_SAKLAMA_GUNU = 3  # defterde olmayan yemekler için


class YemekGunlugu:
    def __init__(self, defter):
        self.defter = defter
        self.kayitlar = []

    def ekle(self, tarih, ogun, yemek):
        if ogun not in OGUNLER:
            print("Böyle bir öğün yok:", ogun)
            return
        self.kayitlar.append({"tarih": tarih, "ogun": ogun, "yemek": yemek})

    def tarifini_bul(self, yemek):
        for tarif in self.defter.tarifler:
            if kucuk_harf(tarif.ad) == kucuk_harf(yemek):
                return tarif
        return None

    def son_tuketim_tarihi(self, kayit):
        tarif = self.tarifini_bul(kayit["yemek"])
        if tarif is not None and tarif.saklama is not None:
            gun = tarif.saklama["en_cok"]
        else:
            gun = VARSAYILAN_SAKLAMA_GUNU
        return kayit["tarih"] + timedelta(days=gun)

    def haftalik_liste(self, gun_icinden):
        # verilen tarihin haftasını pazartesiden başlatıyorum
        pazartesi = gun_icinden - timedelta(days=gun_icinden.weekday())
        hafta = {}
        for i in range(7):
            gun = pazartesi + timedelta(days=i)
            hafta[gun] = [k for k in self.kayitlar if k["tarih"] == gun]
            hafta[gun].sort(key=lambda k: OGUNLER.index(k["ogun"]))
        return hafta

    def haftayi_yazdir(self, gun_icinden):
        for gun, kayitlar in self.haftalik_liste(gun_icinden).items():
            print(f"{GUNLER[gun.weekday()]} ({gun.strftime('%d.%m')})")
            if not kayitlar:
                print("   -")
            for k in kayitlar:
                son = self.son_tuketim_tarihi(k).strftime("%d.%m")
                print(f"   {k['ogun']:9} {k['yemek']:32} artanı en geç {son}")

In [ ]:
# deneme kayıtları
gunluk = YemekGunlugu(defter)
bugun = date.today()
pazartesi = bugun - timedelta(days=bugun.weekday())

gunluk.ekle(pazartesi, "Akşam", "İspir Fasulyesi")
gunluk.ekle(pazartesi, "Ara öğün", "Kakaolu Islak Kek")
gunluk.ekle(pazartesi + timedelta(days=1), "Öğle", "Humus")
gunluk.ekle(pazartesi + timedelta(days=1), "Akşam", "Bolonez Soslu Makarna")
gunluk.ekle(pazartesi + timedelta(days=2), "Kahvaltı", "Menemen")  # defterde olmayan yemek

gunluk.haftayi_yazdir(bugun)

## 7. Arayüzün stil kodları (CSS)

In [ ]:
STIL = """
.defter {
  --zemin: #fff8f2;
  --kart: #ffffff;
  --yazi: #2b2220;
  --soluk: #7a6a63;
  --ana-renk: #cc2e38;
  --ikinci-renk: #f2a65a;
  --cizgi: #f0e2d8;
  --yesil: #2f8f55;
  --turuncu: #c46a12;
  --mavi: #3b7fc4;
  background: var(--zemin);
  color: var(--yazi);
  font-family: system-ui, -apple-system, "Segoe UI", sans-serif;
  border-radius: 16px;
  padding-bottom: 30px;
}
@media (prefers-color-scheme: dark) {
  .defter {
    --zemin: #1b1514;
    --kart: #271f1d;
    --yazi: #f3e9e4;
    --soluk: #b5a39b;
    --ana-renk: #ff6b73;
    --cizgi: #3a2e2b;
    --yesil: #5fcf8b;
    --turuncu: #f0a55a;
    --mavi: #7ab6f5;
  }
}
.defter * { box-sizing: border-box; }
.defter [hidden] { display: none !important; }

/* sekmeler */
.defter .sekmeler { display: flex; flex-wrap: wrap; gap: 8px; max-width: 1100px; margin: auto; padding: 18px 20px 0; }
.defter .sekme {
  padding: 9px 16px; border-radius: 99px; cursor: pointer; font-size: .95rem;
  border: 1px solid var(--cizgi); background: var(--kart); color: var(--yazi);
}
.defter .sekme.secili { background: var(--yazi); border-color: var(--yazi); color: var(--zemin); }

/* üst kısım */
.defter header { padding: 20px 20px 8px; max-width: 1100px; margin: auto; }
.defter h1 { margin: 0; font-size: 1.8rem; }
.defter h1 span { color: var(--ana-renk); }
.defter #sayi, .defter .aciklama { color: var(--soluk); margin: 4px 0 14px; }
.defter input, .defter select {
  width: 100%; padding: 12px 16px; font-size: 1rem; font-family: inherit;
  border-radius: 12px; border: 1px solid var(--cizgi);
  background: var(--kart); color: var(--yazi);
}
.defter input[type=checkbox] { width: auto; }
.defter .kategoriler { display: flex; gap: 8px; overflow-x: auto; padding: 12px 0 2px; }
.defter .kategori {
  flex: none; padding: 7px 14px; border-radius: 99px; cursor: pointer;
  border: 1px solid var(--cizgi); background: var(--kart); color: var(--yazi); font-size: .88rem;
}
.defter .kategori.secili { background: var(--ana-renk); border-color: var(--ana-renk); color: #fff; }
.defter .kucuk-buton {
  flex: none; padding: 10px 18px; border: 0; border-radius: 12px; cursor: pointer;
  background: var(--ana-renk); color: #fff; font-size: .95rem; font-family: inherit;
}
.defter .kucuk-buton.ikincil { background: var(--kart); color: var(--yazi); border: 1px solid var(--cizgi); }

/* tarif kartları */
.defter .kartlar {
  max-width: 1100px; margin: auto; padding: 12px 20px;
  display: grid; grid-template-columns: repeat(auto-fill, minmax(230px, 1fr)); gap: 18px;
}
.defter .kart {
  background: var(--kart); border: 1px solid var(--cizgi); border-radius: 16px;
  overflow: hidden; display: flex; flex-direction: column;
}
.defter .kart h2 { font-size: 1.02rem; margin: 12px 14px 6px; }
.defter .resim {
  height: 150px; position: relative; overflow: hidden;
  background: linear-gradient(135deg, var(--ana-renk), var(--ikinci-renk));
  display: flex; align-items: center; justify-content: center;
}
.defter .resim .emoji { font-size: 3rem; }
.defter .resim img { position: absolute; inset: 0; width: 100%; height: 100%; object-fit: cover; }
.defter .etiketler { display: flex; flex-wrap: wrap; gap: 6px; padding: 0 14px 6px; font-size: .8rem; color: var(--soluk); }
.defter .etiketler span { background: var(--zemin); padding: 3px 8px; border-radius: 99px; }
.defter .buton {
  margin: auto 14px 14px; padding: 10px; border: 0; border-radius: 10px; cursor: pointer;
  background: var(--ana-renk); color: #fff; font-size: .95rem;
}
.defter .bos { grid-column: 1 / -1; text-align: center; color: var(--soluk); padding: 40px 0; }
.defter .ozet { grid-column: 1 / -1; margin: 0; color: var(--soluk); }

/* buzdolabı */
.defter .ekle-satiri { display: flex; gap: 8px; }
.defter .uyari-yazi { color: var(--turuncu); font-size: .85rem; min-height: 1.2em; margin: 6px 0 0; }
.defter .cipler { display: flex; flex-wrap: wrap; gap: 6px; }
.defter .cip {
  padding: 6px 12px; border-radius: 99px; cursor: pointer; font-size: .85rem; font-family: inherit;
  border: 1px solid var(--cizgi); background: var(--kart); color: var(--yazi);
}
.defter .cip.secili { background: var(--yesil); border-color: var(--yesil); color: #fff; }
.defter .cip.secili::before { content: "✓ "; }
.defter .secilenler { display: flex; flex-wrap: wrap; gap: 6px; align-items: center; margin-top: 12px; }
.defter .secilenler .cip::after { content: " ×"; }
.defter .secilenler .cip.secili::before { content: ""; }
.defter .temizle { border: 0; background: none; color: var(--soluk); cursor: pointer; text-decoration: underline; font-size: .85rem; }
.defter .kutu { background: var(--kart); border: 1px solid var(--cizgi); border-radius: 16px; padding: 12px 16px; margin-top: 12px; }
.defter .kutu summary { cursor: pointer; font-weight: 600; }
.defter .kutu h3 { margin: 0 0 8px; font-size: 1rem; }
.defter .kutu h4 { margin: 14px 0 6px; font-size: .78rem; color: var(--soluk); text-transform: uppercase; letter-spacing: .04em; }
.defter .uyum { padding: 0 14px 10px; font-size: .82rem; }
.defter .cubuk { height: 6px; background: var(--cizgi); border-radius: 99px; overflow: hidden; margin-bottom: 6px; }
.defter .cubuk span { display: block; height: 100%; background: var(--yesil); }
.defter .eksik { color: var(--soluk); }
.defter .tamam { color: var(--yesil); font-weight: 600; }
.defter .satir { display: flex; align-items: center; gap: 10px; padding: 8px 0; border-bottom: 1px dashed var(--cizgi); }
.defter .satir:last-child { border-bottom: 0; }
.defter .satir .ad { flex: 1; }
.defter .satir small { display: block; color: var(--soluk); }
.defter .durum { font-size: .8rem; padding: 3px 9px; border-radius: 99px; background: var(--zemin); white-space: nowrap; }
.defter .durum.iyi { color: var(--yesil); }
.defter .durum.uyari { color: var(--turuncu); }
.defter .durum.gecti { color: var(--ana-renk); }
.defter .sil { border: 0; background: none; color: var(--soluk); cursor: pointer; font-size: 1rem; padding: 2px 6px; }
.defter .tarif-link { color: var(--ana-renk); cursor: pointer; text-decoration: underline dotted; }

/* yemek günlüğü */
.defter .gunluk-formu { display: grid; grid-template-columns: 160px 140px 1fr auto auto; gap: 8px; align-items: center; }
@media (max-width: 760px) {
  .defter .gunluk-formu { grid-template-columns: 1fr 1fr; }
  .defter .gunluk-formu .genis { grid-column: 1 / -1; }
}
.defter .onay { display: flex; align-items: center; gap: 6px; font-size: .85rem; color: var(--soluk); white-space: nowrap; cursor: pointer; }
.defter .hafta-gezgini { display: flex; align-items: center; gap: 8px; flex-wrap: wrap; margin-top: 18px; }
.defter .hafta-gezgini b { min-width: 190px; text-align: center; }
.defter #hafta-ozeti { color: var(--soluk); margin: 10px 0 0; }
.defter .hafta {
  max-width: 1100px; margin: auto; padding: 12px 20px;
  display: grid; grid-template-columns: repeat(auto-fill, minmax(210px, 1fr)); gap: 12px;
}
.defter .gun { background: var(--kart); border: 1px solid var(--cizgi); border-radius: 14px; padding: 12px 14px; }
.defter .gun.bugun { border-color: var(--ana-renk); box-shadow: inset 0 0 0 1px var(--ana-renk); }
.defter .gun h3 { margin: 0 0 6px; font-size: .95rem; display: flex; justify-content: space-between; }
.defter .gun h3 small { color: var(--soluk); font-weight: 400; }
.defter .gun ul { list-style: none; padding: 0; margin: 0; }
.defter .gun li { display: flex; align-items: center; gap: 6px; padding: 5px 0; font-size: .9rem; border-bottom: 1px dashed var(--cizgi); }
.defter .gun li:last-child { border-bottom: 0; }
.defter .gun li .yemek-adi { flex: 1; }
.defter .ogun { flex: none; font-size: .7rem; background: var(--zemin); color: var(--soluk); padding: 2px 7px; border-radius: 99px; }
.defter .bos-gun { color: var(--soluk); font-size: .85rem; }

/* tarif detay sayfası */
.defter #detay { max-width: 860px; margin: auto; padding: 20px; }
.defter .geri {
  border: 1px solid var(--cizgi); background: var(--kart); color: var(--yazi);
  padding: 8px 16px; border-radius: 99px; cursor: pointer; font-size: .9rem; margin-bottom: 16px;
}
.defter #detay .resim { height: 260px; border-radius: 16px; }
.defter #detay .resim .emoji { font-size: 5rem; }
.defter .kategori-adi { color: var(--ana-renk); font-weight: 600; font-size: .85rem; margin: 18px 0 4px; }
.defter #detay h2 { margin: 0 0 14px; font-size: 1.6rem; }
.defter .bilgi-kutulari { display: grid; grid-template-columns: repeat(4, 1fr); gap: 10px; }
@media (max-width: 680px) { .defter .bilgi-kutulari { grid-template-columns: repeat(2, 1fr); } }
.defter .bilgi-kutusu {
  background: var(--kart); border: 1px solid var(--cizgi); border-radius: 12px;
  padding: 10px; text-align: center;
}
.defter .bilgi-kutusu small { display: block; color: var(--soluk); font-size: .78rem; }
.defter .bilgi-kutusu b { font-size: 1rem; }
.defter .saklama-kutusu {
  margin-top: 12px; padding: 12px 14px; border-radius: 12px; line-height: 1.5; font-size: .92rem;
  background: var(--kart); border: 1px solid var(--cizgi); border-left: 4px solid var(--mavi);
}
.defter .detay-butonlari { display: flex; flex-wrap: wrap; gap: 8px; margin-top: 12px; }
.defter .mesaj { color: var(--yesil); font-size: .88rem; min-height: 1.2em; margin: 8px 0 0; }
.defter .bolumler { display: grid; grid-template-columns: 1fr 1.4fr; gap: 18px; margin-top: 18px; }
@media (max-width: 680px) { .defter .bolumler { grid-template-columns: 1fr; } }
.defter .bolum { background: var(--kart); border: 1px solid var(--cizgi); border-radius: 16px; padding: 16px 18px; }
.defter .bolum h3 { margin: 0 0 10px; color: var(--ana-renk); }
.defter .malzemeler { list-style: none; padding: 0; margin: 0; }
.defter .malzemeler li { padding: 7px 0; border-bottom: 1px dashed var(--cizgi); cursor: pointer; }
.defter .malzemeler li:last-child { border-bottom: 0; }
.defter .malzemeler li::before { content: "○ "; color: var(--ana-renk); }
.defter .malzemeler li.alindi { text-decoration: line-through; color: var(--soluk); }
.defter .malzemeler li.alindi::before { content: "● "; }
.defter .adimlar { margin: 0; padding-left: 22px; }
.defter .adimlar li { margin: 0 0 10px; line-height: 1.5; }
.defter .not { color: var(--soluk); font-size: .8rem; margin-top: 8px; }
"""

## 8. Arayüzün çalışma kodları (JavaScript)
Arama, kategoriler, **Tarifi Göster**, **Buzdolabım** önerileri ve **Yediklerim** günlüğü burada çalışıyor.
Dolaptaki malzemeler ve günlük kayıtları tarayıcının hafızasında (`localStorage`) tutuluyor.

In [ ]:
SCRIPT = """
const tarifler = __TARIFLER__;
const kategoriler = __KATEGORILER__;
const malzemeGruplari = __MALZEMELER__;
const OGUNLER = __OGUNLER__;
const GUNLER = __GUNLER__;
const VARSAYILAN_SAKLAMA_GUNU = __VARSAYILAN_GUN__;
const AYLAR = ["Ocak", "Şubat", "Mart", "Nisan", "Mayıs", "Haziran", "Temmuz",
               "Ağustos", "Eylül", "Ekim", "Kasım", "Aralık"];

let seciliKategori = "";
let aramaMetni = "";
let aktifSekme = "tarifler";
let acikTarifNo = null;
let haftaKaydirma = 0;  // 0 = bu hafta, -1 = geçen hafta ...

// ---------- tarayıcıda saklama (sayfa kapanınca kaybolmasın) ----------
function oku(anahtar, varsayilan) {
  try {
    const deger = localStorage.getItem(anahtar);
    return deger ? JSON.parse(deger) : varsayilan;
  } catch (e) {
    return varsayilan;
  }
}

function kaydet(anahtar, deger) {
  try { localStorage.setItem(anahtar, JSON.stringify(deger)); } catch (e) {}
}

let dolaptakiler = oku("tarif-defteri-dolap", []);
let gunluk = oku("tarif-defteri-gunluk", []);
let hazirYemekler = oku("tarif-defteri-hazir", []);

const sayfalar = {
  tarifler: document.getElementById("sayfa-tarifler"),
  dolap: document.getElementById("sayfa-dolap"),
  gunluk: document.getElementById("sayfa-gunluk"),
};
const liste = document.getElementById("liste");
const detay = document.getElementById("detay");

// ---------- yardımcı fonksiyonlar ----------
function kucukHarf(yazi) {
  return yazi.toLocaleLowerCase("tr");
}

function guvenli(yazi) {
  // kullanıcının yazdığı metin html gibi çalışmasın
  const karsilik = {"&": "&amp;", "<": "&lt;", ">": "&gt;", '"': "&quot;", "'": "&#39;"};
  return String(yazi).replace(/[&<>"']/g, k => karsilik[k]);
}

function sureYazisi(saklama) {
  if (!saklama) return "Bilgi yok";
  if (saklama.en_az >= 30) return Math.floor(saklama.en_az / 30) + "-" + Math.floor(saklama.en_cok / 30) + " ay";
  if (saklama.en_az === saklama.en_cok) return saklama.en_az + " gün";
  return saklama.en_az + "-" + saklama.en_cok + " gün";
}

function tarihYazisi(tarih) {
  // Date -> "2026-10-06"
  const ay = String(tarih.getMonth() + 1).padStart(2, "0");
  const gun = String(tarih.getDate()).padStart(2, "0");
  return tarih.getFullYear() + "-" + ay + "-" + gun;
}

function tarihNesnesi(yazi) {
  const [yil, ay, gun] = yazi.split("-").map(Number);
  return new Date(yil, ay - 1, gun);
}

function gunEkle(yazi, gunSayisi) {
  const tarih = tarihNesnesi(yazi);
  tarih.setDate(tarih.getDate() + gunSayisi);
  return tarihYazisi(tarih);
}

function gunFarki(ilk, son) {
  return Math.round((tarihNesnesi(son) - tarihNesnesi(ilk)) / 86400000);
}

function guzelTarih(yazi) {
  const tarih = tarihNesnesi(yazi);
  return tarih.getDate() + " " + AYLAR[tarih.getMonth()];
}

function bugun() {
  return tarihYazisi(new Date());
}

function ogunTahmini() {
  const saat = new Date().getHours();
  if (saat < 11) return "Kahvaltı";
  if (saat < 16) return "Öğle";
  if (saat < 22) return "Akşam";
  return "Ara öğün";
}

function tarifNumarasi(yemekAdi) {
  return tarifler.findIndex(t => kucukHarf(t.ad) === kucukHarf(yemekAdi.trim()));
}

// ---------- sekmeler ----------
function sekmeyeGec(ad) {
  aktifSekme = ad;
  document.querySelectorAll(".sekme").forEach(b => b.classList.toggle("secili", b.dataset.sekme === ad));
  for (const isim in sayfalar) sayfalar[isim].hidden = isim !== ad;
  detay.hidden = true;
  if (ad === "dolap") dolabiCiz();
  if (ad === "gunluk") gunluguCiz();
}

// ---------- tarif listesi ----------
function resimKutusu(tarif) {
  let html = `<div class="resim"><span class="emoji">${tarif.emoji}</span>`;
  if (tarif.resim) {
    html += `<img src="${tarif.resim}" alt="${tarif.ad}" loading="lazy" referrerpolicy="no-referrer" onerror="this.remove()">`;
  }
  return html + "</div>";
}

function etiketler(tarif) {
  return `<div class="etiketler">
    <span>👥 ${tarif.kisi} kişilik</span>
    <span>⏱ ${tarif.hazirlik}</span>
    <span>🔥 ${tarif.pisirme}</span>
    <span>🧊 ${sureYazisi(tarif.saklama)}</span>
  </div>`;
}

function kartHtml(tarif, no, ekBilgi = "") {
  return `<article class="kart">
    ${resimKutusu(tarif)}
    <h2>${tarif.ad}</h2>
    ${etiketler(tarif)}
    ${ekBilgi}
    <button class="buton" data-no="${no}">Tarifi Göster</button>
  </article>`;
}

function kategoriButonlariniYap() {
  let html = `<button class="kategori secili" data-kod="">Tümü</button>`;
  for (const kod in kategoriler) {
    html += `<button class="kategori" data-kod="${kod}">${kategoriler[kod]}</button>`;
  }
  document.getElementById("kategoriler").innerHTML = html;
}

function listeyiCiz() {
  let html = "";
  let bulunan = 0;

  tarifler.forEach((tarif, no) => {
    const kategoriUygun = seciliKategori === "" || tarif.kategori === seciliKategori;
    const aramaUygun = kucukHarf(tarif.ad).includes(kucukHarf(aramaMetni));
    if (kategoriUygun && aramaUygun) {
      bulunan++;
      html += kartHtml(tarif, no);
    }
  });

  if (bulunan === 0) {
    html = `<div class="bos">Sonuç bulunamadı 🍳</div>`;
  }
  liste.innerHTML = html;
  document.getElementById("sayi").textContent = tarifler.length + " tarif · " + bulunan + " sonuç";
}

// ---------- tarif detayı ----------
function tarifiAc(no) {
  const tarif = tarifler[no];
  acikTarifNo = no;

  let malzemeler = "";
  tarif.malzemeler.forEach(m => { malzemeler += `<li>${m}</li>`; });

  let adimlar = "";
  tarif.yapilis.forEach(a => { adimlar += `<li>${a}</li>`; });

  let saklama = "";
  if (tarif.saklama) {
    saklama = `<div class="saklama-kutusu">🧊 <b>Buzdolabında ortalama ${sureYazisi(tarif.saklama)}</b> dayanır.<br>${tarif.saklama.not}</div>`;
  }

  detay.innerHTML = `
    <button class="geri">← Geri dön</button>
    ${resimKutusu(tarif)}
    <p class="kategori-adi">${kategoriler[tarif.kategori]}</p>
    <h2>${tarif.emoji} ${tarif.ad}</h2>
    <div class="bilgi-kutulari">
      <div class="bilgi-kutusu"><small>Kişi sayısı</small><b>👥 ${tarif.kisi}</b></div>
      <div class="bilgi-kutusu"><small>Hazırlık</small><b>⏱ ${tarif.hazirlik}</b></div>
      <div class="bilgi-kutusu"><small>Pişirme</small><b>🔥 ${tarif.pisirme}</b></div>
      <div class="bilgi-kutusu"><small>Dolapta saklama</small><b>🧊 ${sureYazisi(tarif.saklama)}</b></div>
    </div>
    ${saklama}
    <div class="detay-butonlari">
      <button class="kucuk-buton yedim">🍽 Bugün bunu yedim</button>
      <button class="kucuk-buton ikincil dolaba-kaldir">🧊 Artanı dolaba kaldırdım</button>
    </div>
    <p class="mesaj" id="detay-mesaj"></p>
    <div class="bolumler">
      <div class="bolum">
        <h3>Malzemeler</h3>
        <ul class="malzemeler">${malzemeler}</ul>
        <p class="not">Hazırladığın malzemenin üstüne tıklayıp işaretleyebilirsin.</p>
      </div>
      <div class="bolum">
        <h3>Yapılışı</h3>
        <ol class="adimlar">${adimlar}</ol>
      </div>
    </div>`;

  sayfalar[aktifSekme].hidden = true;
  detay.hidden = false;
  detay.scrollIntoView();
}

function geriDon() {
  detay.hidden = true;
  sayfalar[aktifSekme].hidden = false;
  if (aktifSekme === "dolap") dolabiCiz();
  if (aktifSekme === "gunluk") gunluguCiz();
  sayfalar[aktifSekme].scrollIntoView();
}

// ---------- buzdolabım ----------
function tumMalzemeler() {
  return Object.values(malzemeGruplari).flat();
}

function malzemeDegistir(malzeme) {
  if (dolaptakiler.includes(malzeme)) {
    dolaptakiler = dolaptakiler.filter(m => m !== malzeme);
  } else {
    dolaptakiler.push(malzeme);
  }
  kaydet("tarif-defteri-dolap", dolaptakiler);
  dolabiCiz();
}

function yazilanMalzemeyiEkle() {
  const kutu = document.getElementById("malzeme-yaz");
  const uyari = document.getElementById("malzeme-uyari");
  const malzeme = kucukHarf(kutu.value.trim());
  if (malzeme === "") return;

  if (!tumMalzemeler().includes(malzeme)) {
    uyari.textContent = `"${malzeme}" tariflerimizde geçmiyor. Listeden seçmeyi dene.`;
    return;
  }
  uyari.textContent = "";
  kutu.value = "";
  if (!dolaptakiler.includes(malzeme)) {
    dolaptakiler.push(malzeme);
    kaydet("tarif-defteri-dolap", dolaptakiler);
  }
  dolabiCiz();
}

function malzemeleriCiz() {
  let secilenler = "";
  dolaptakiler.forEach(m => {
    secilenler += `<button class="cip secili" data-malzeme="${m}" title="Çıkar">${m}</button>`;
  });
  if (dolaptakiler.length > 0) {
    secilenler = `<b>Dolabımda:</b> ${secilenler} <button class="temizle" id="dolabi-bosalt">Hepsini temizle</button>`;
  } else {
    secilenler = `<span class="eksik">Dolabın şu an boş görünüyor. Aşağıdan malzeme seç.</span>`;
  }
  document.getElementById("secilenler").innerHTML = secilenler;

  let gruplar = "";
  for (const grup in malzemeGruplari) {
    gruplar += `<h4>${grup}</h4><div class="cipler">`;
    malzemeGruplari[grup].forEach(m => {
      const secili = dolaptakiler.includes(m) ? " secili" : "";
      gruplar += `<button class="cip${secili}" data-malzeme="${m}">${m}</button>`;
    });
    gruplar += "</div>";
  }
  document.getElementById("malzeme-gruplari").innerHTML = gruplar;
}

function onerileriCiz() {
  const kutu = document.getElementById("oneriler");
  if (dolaptakiler.length === 0) {
    kutu.innerHTML = `<div class="bos">Dolabındaki malzemeleri seçince yapabileceğin tarifler burada çıkacak 🥕</div>`;
    return;
  }

  const sonuc = [];
  tarifler.forEach((tarif, no) => {
    const olanlar = tarif.anahtarlar.filter(m => dolaptakiler.includes(m));
    const eksikler = tarif.anahtarlar.filter(m => !dolaptakiler.includes(m));
    if (olanlar.length === 0) return;
    sonuc.push({tarif, no, eksikler, uyum: olanlar.length / tarif.anahtarlar.length});
  });
  // önce uyumu yüksek olan, eşitse eksiği az olan
  sonuc.sort((a, b) => b.uyum - a.uyum || a.eksikler.length - b.eksikler.length);

  if (sonuc.length === 0) {
    kutu.innerHTML = `<div class="bos">Bu malzemelerle eşleşen tarif bulamadım 🍳</div>`;
    return;
  }

  const hemenYapilir = sonuc.filter(s => s.eksikler.length === 0).length;
  let html = `<p class="ozet">${sonuc.length} tarif bulundu · ${hemenYapilir} tanesinin bütün malzemeleri dolabında</p>`;
  sonuc.forEach(s => {
    const yuzde = Math.round(s.uyum * 100);
    let eksikYazi;
    if (s.eksikler.length === 0) {
      eksikYazi = `<span class="tamam">✅ Bütün malzemeler dolabında</span>`;
    } else {
      eksikYazi = `<span class="eksik">Eksik: ${s.eksikler.join(", ")}</span>`;
    }
    const uyum = `<div class="uyum">
      <div class="cubuk"><span style="width:${yuzde}%"></span></div>
      <b>%${yuzde} uyum</b> · ${eksikYazi}
    </div>`;
    html += kartHtml(s.tarif, s.no, uyum);
  });
  kutu.innerHTML = html;
}

// dolaba kaldırılan pişmiş yemekler
function hazirYemekEkle(yemek, tarih) {
  const no = tarifNumarasi(yemek);
  let gun = VARSAYILAN_SAKLAMA_GUNU;
  if (no >= 0 && tarifler[no].saklama) gun = tarifler[no].saklama.en_cok;
  hazirYemekler.push({id: Date.now() + Math.random(), yemek: yemek, tarih: tarih, sonGun: gunEkle(tarih, gun)});
  kaydet("tarif-defteri-hazir", hazirYemekler);
}

function hazirYemekleriCiz() {
  const kutu = document.getElementById("hazir-yemekler");
  if (hazirYemekler.length === 0) {
    kutu.innerHTML = `<h3>🍱 Dolaptaki pişmiş yemekler</h3>
      <p class="eksik">Henüz yok. Bir tarifte "Artanı dolaba kaldırdım" dediğinde burada ne zamana kadar yemen gerektiği görünür.</p>`;
    return;
  }

  const sirali = [...hazirYemekler].sort((a, b) => a.sonGun.localeCompare(b.sonGun));
  let html = `<h3>🍱 Dolaptaki pişmiş yemekler</h3>`;
  sirali.forEach(y => {
    const kalan = gunFarki(bugun(), y.sonGun);
    let durum;
    if (kalan < 0) durum = `<span class="durum gecti">⛔ Süresi geçti</span>`;
    else if (kalan === 0) durum = `<span class="durum uyari">⚠️ Bugün son gün</span>`;
    else if (kalan === 1) durum = `<span class="durum uyari">⚠️ Yarın son gün</span>`;
    else durum = `<span class="durum iyi">✅ ${kalan} gün kaldı</span>`;

    const no = tarifNumarasi(y.yemek);
    const ad = no >= 0 ? `<span class="tarif-link" data-no="${no}">${guvenli(y.yemek)}</span>` : guvenli(y.yemek);
    html += `<div class="satir">
      <div class="ad">${ad}<small>${guzelTarih(y.tarih)} tarihinde kaldırıldı · en geç ${guzelTarih(y.sonGun)}</small></div>
      ${durum}
      <button class="sil" data-hazir="${y.id}" title="Bitti / attım">✕</button>
    </div>`;
  });
  kutu.innerHTML = html;
}

function dolabiCiz() {
  malzemeleriCiz();
  hazirYemekleriCiz();
  onerileriCiz();
}

// ---------- yemek günlüğü ----------
function gunlugeEkle(tarih, ogun, yemek) {
  gunluk.push({id: Date.now() + Math.random(), tarih: tarih, ogun: ogun, yemek: yemek});
  kaydet("tarif-defteri-gunluk", gunluk);
}

function haftaninGunleri() {
  const simdi = new Date();
  const pazartesi = new Date(simdi.getFullYear(), simdi.getMonth(), simdi.getDate());
  pazartesi.setDate(pazartesi.getDate() - (simdi.getDay() + 6) % 7 + haftaKaydirma * 7);
  const gunler = [];
  for (let i = 0; i < 7; i++) {
    const gun = new Date(pazartesi);
    gun.setDate(pazartesi.getDate() + i);
    gunler.push(tarihYazisi(gun));
  }
  return gunler;
}

function gunluguCiz() {
  const gunler = haftaninGunleri();
  document.getElementById("hafta-adi").textContent =
    guzelTarih(gunler[0]) + " – " + guzelTarih(gunler[6]) + " " + tarihNesnesi(gunler[6]).getFullYear();

  let html = "";
  let toplam = 0;
  const sayac = {};

  gunler.forEach((gun, i) => {
    const kayitlar = gunluk
      .filter(k => k.tarih === gun)
      .sort((a, b) => OGUNLER.indexOf(a.ogun) - OGUNLER.indexOf(b.ogun));
    toplam += kayitlar.length;

    let satirlar = "";
    kayitlar.forEach(k => {
      sayac[k.yemek] = (sayac[k.yemek] || 0) + 1;
      const no = tarifNumarasi(k.yemek);
      const ad = no >= 0
        ? `<span class="tarif-link" data-no="${no}">${tarifler[no].emoji} ${guvenli(k.yemek)}</span>`
        : guvenli(k.yemek);
      satirlar += `<li><span class="ogun">${k.ogun}</span><span class="yemek-adi">${ad}</span>
        <button class="sil" data-kayit="${k.id}" title="Sil">✕</button></li>`;
    });
    if (satirlar === "") satirlar = `<li class="bos-gun">Kayıt yok</li>`;

    const bugunMu = gun === bugun() ? " bugun" : "";
    html += `<div class="gun${bugunMu}">
      <h3>${GUNLER[i]} <small>${guzelTarih(gun)}</small></h3>
      <ul>${satirlar}</ul>
    </div>`;
  });
  document.getElementById("hafta").innerHTML = html;

  let ozet = "Bu hafta için henüz kayıt yok.";
  if (toplam > 0) {
    const enCok = Object.keys(sayac).sort((a, b) => sayac[b] - sayac[a])[0];
    ozet = `Bu hafta ${toplam} öğün kaydedildi · ${Object.keys(sayac).length} farklı yemek · En çok yenen: ${guvenli(enCok)} (${sayac[enCok]} kez)`;
  }
  document.getElementById("hafta-ozeti").innerHTML = ozet;
}

function gunlukFormunuHazirla() {
  document.getElementById("g-tarih").value = bugun();
  let secenekler = "";
  OGUNLER.forEach(o => { secenekler += `<option>${o}</option>`; });
  document.getElementById("g-ogun").innerHTML = secenekler;
  document.getElementById("g-ogun").value = ogunTahmini();

  let adlar = "";
  tarifler.forEach(t => { adlar += `<option value="${t.ad}">`; });
  document.getElementById("tarif-adlari").innerHTML = adlar;

  let malzemeler = "";
  tumMalzemeler().forEach(m => { malzemeler += `<option value="${m}">`; });
  document.getElementById("malzeme-onerileri").innerHTML = malzemeler;
}

// ---------- butonlar ----------
document.getElementById("sekmeler").addEventListener("click", e => {
  const buton = e.target.closest(".sekme");
  if (buton) sekmeyeGec(buton.dataset.sekme);
});

document.getElementById("kategoriler").addEventListener("click", e => {
  const buton = e.target.closest(".kategori");
  if (!buton) return;
  seciliKategori = buton.dataset.kod;
  document.querySelectorAll(".kategori").forEach(b => b.classList.toggle("secili", b === buton));
  listeyiCiz();
});

document.getElementById("ara").addEventListener("input", e => {
  aramaMetni = e.target.value;
  listeyiCiz();
});

// "Tarifi Göster" butonları ve tarif adlarına tıklama bütün sayfalarda aynı
document.querySelector(".defter").addEventListener("click", e => {
  const buton = e.target.closest(".buton, .tarif-link");
  if (buton && !detay.contains(buton)) tarifiAc(Number(buton.dataset.no));
});

detay.addEventListener("click", e => {
  if (e.target.closest(".geri")) {
    geriDon();
    return;
  }
  const tarif = tarifler[acikTarifNo];
  const mesaj = document.getElementById("detay-mesaj");
  if (e.target.closest(".yedim")) {
    const ogun = ogunTahmini();
    gunlugeEkle(bugun(), ogun, tarif.ad);
    mesaj.textContent = `✓ Günlüğüne eklendi (bugün, ${ogun}).`;
    return;
  }
  if (e.target.closest(".dolaba-kaldir")) {
    hazirYemekEkle(tarif.ad, bugun());
    const son = hazirYemekler[hazirYemekler.length - 1].sonGun;
    mesaj.textContent = `✓ Buzdolabına eklendi. En geç ${guzelTarih(son)} tarihine kadar tüket.`;
    return;
  }
  const malzeme = e.target.closest(".malzemeler li");
  if (malzeme) malzeme.classList.toggle("alindi");
});

sayfalar.dolap.addEventListener("click", e => {
  if (e.target.id === "malzeme-ekle-buton") {
    yazilanMalzemeyiEkle();
    return;
  }
  if (e.target.id === "dolabi-bosalt") {
    dolaptakiler = [];
    kaydet("tarif-defteri-dolap", dolaptakiler);
    dolabiCiz();
    return;
  }
  const cip = e.target.closest(".cip");
  if (cip) {
    malzemeDegistir(cip.dataset.malzeme);
    return;
  }
  const sil = e.target.closest("[data-hazir]");
  if (sil) {
    hazirYemekler = hazirYemekler.filter(y => String(y.id) !== sil.dataset.hazir);
    kaydet("tarif-defteri-hazir", hazirYemekler);
    hazirYemekleriCiz();
  }
});

document.getElementById("malzeme-yaz").addEventListener("keydown", e => {
  if (e.key === "Enter") yazilanMalzemeyiEkle();
});

document.getElementById("gunluk-formu").addEventListener("submit", e => {
  e.preventDefault();
  const tarih = document.getElementById("g-tarih").value || bugun();
  const ogun = document.getElementById("g-ogun").value;
  const yemekKutusu = document.getElementById("g-yemek");
  const yemek = yemekKutusu.value.trim();
  if (yemek === "") return;

  // tarif defterde varsa adını defterdeki gibi yaz
  const no = tarifNumarasi(yemek);
  const ad = no >= 0 ? tarifler[no].ad : yemek;
  gunlugeEkle(tarih, ogun, ad);
  if (document.getElementById("g-dolap").checked) hazirYemekEkle(ad, tarih);

  yemekKutusu.value = "";
  document.getElementById("g-dolap").checked = false;
  // eklenen günün haftasına git
  const fark = gunFarki(haftaninGunleri()[0], tarih);
  haftaKaydirma += Math.floor(fark / 7);
  gunluguCiz();
});

sayfalar.gunluk.addEventListener("click", e => {
  if (e.target.id === "onceki-hafta") haftaKaydirma--;
  else if (e.target.id === "sonraki-hafta") haftaKaydirma++;
  else if (e.target.id === "bu-hafta") haftaKaydirma = 0;
  else {
    const sil = e.target.closest("[data-kayit]");
    if (!sil) return;
    gunluk = gunluk.filter(k => String(k.id) !== sil.dataset.kayit);
    kaydet("tarif-defteri-gunluk", gunluk);
  }
  gunluguCiz();
});

kategoriButonlariniYap();
gunlukFormunuHazirla();
listeyiCiz();
"""

## 9. Arayuz sınıfı
Defterdeki tarifleri, malzeme gruplarını ve öğünleri JSON'a çevirip JavaScript koduna yerleştiriyor, sonra CSS ve HTML ile birleştirip ekranda gösteriyor.

In [ ]:
class Arayuz:
    def __init__(self, defter):
        self.defter = defter

    def sayfa_iskeleti(self):
        return """
        <div class="defter">
          <nav class="sekmeler" id="sekmeler">
            <button class="sekme secili" data-sekme="tarifler">📖 Tarifler</button>
            <button class="sekme" data-sekme="dolap">🧊 Buzdolabım</button>
            <button class="sekme" data-sekme="gunluk">📅 Yediklerim</button>
          </nav>

          <div id="sayfa-tarifler">
            <header>
              <h1>🍲 Tarif <span>Defteri</span></h1>
              <p id="sayi"></p>
              <input id="ara" placeholder="Tarif ara... (örn: kek, çorba)">
              <div class="kategoriler" id="kategoriler"></div>
            </header>
            <main id="liste" class="kartlar"></main>
          </div>

          <div id="sayfa-dolap" hidden>
            <header>
              <h1>🧊 Buzdolabım</h1>
              <p class="aciklama">Dolabında olan malzemeleri seç, elindekilerle yapabileceğin tarifleri bulalım.</p>
              <div class="ekle-satiri">
                <input id="malzeme-yaz" list="malzeme-onerileri" placeholder="Malzeme yaz... (örn: yumurta)">
                <button class="kucuk-buton" id="malzeme-ekle-buton">Ekle</button>
              </div>
              <datalist id="malzeme-onerileri"></datalist>
              <p class="uyari-yazi" id="malzeme-uyari"></p>
              <div class="secilenler" id="secilenler"></div>
              <details class="kutu">
                <summary>Bütün malzemeler (seçmek için tıkla)</summary>
                <div id="malzeme-gruplari"></div>
              </details>
              <div class="kutu" id="hazir-yemekler"></div>
            </header>
            <main id="oneriler" class="kartlar"></main>
          </div>

          <div id="sayfa-gunluk" hidden>
            <header>
              <h1>📅 Yemek <span>Günlüğüm</span></h1>
              <p class="aciklama">Her gün yediklerini ekle, haftalık listen burada birikir.</p>
              <form class="gunluk-formu" id="gunluk-formu">
                <input type="date" id="g-tarih">
                <select id="g-ogun"></select>
                <input class="genis" id="g-yemek" list="tarif-adlari" placeholder="Ne yedin? (örn: Humus)" autocomplete="off">
                <label class="onay"><input type="checkbox" id="g-dolap"> Artanı dolaba kaldırdım</label>
                <button class="kucuk-buton" type="submit">Ekle</button>
              </form>
              <datalist id="tarif-adlari"></datalist>
              <div class="hafta-gezgini">
                <button class="kucuk-buton ikincil" id="onceki-hafta">← Önceki</button>
                <b id="hafta-adi"></b>
                <button class="kucuk-buton ikincil" id="sonraki-hafta">Sonraki →</button>
                <button class="kucuk-buton ikincil" id="bu-hafta">Bu hafta</button>
              </div>
              <p id="hafta-ozeti"></p>
            </header>
            <main id="hafta" class="hafta"></main>
          </div>

          <section id="detay" hidden></section>
        </div>
        """

    def script_hazirla(self):
        # "</" yazısı script etiketini erken kapatmasın diye
        tarif_verisi = self.defter.json_verisi().replace("</", "<\\/")
        kategori_verisi = json.dumps(self.defter.kategoriler, ensure_ascii=False)
        malzeme_verisi = json.dumps(MALZEME_GRUPLARI, ensure_ascii=False)
        return (SCRIPT
                .replace("__TARIFLER__", tarif_verisi)
                .replace("__KATEGORILER__", kategori_verisi)
                .replace("__MALZEMELER__", malzeme_verisi)
                .replace("__OGUNLER__", json.dumps(OGUNLER, ensure_ascii=False))
                .replace("__GUNLER__", json.dumps(GUNLER, ensure_ascii=False))
                .replace("__VARSAYILAN_GUN__", str(VARSAYILAN_SAKLAMA_GUNU)))

    def html_olustur(self):
        return (
            "<style>" + STIL + "</style>"
            + self.sayfa_iskeleti()
            + "<script>" + self.script_hazirla() + "</script>"
        )

    def goster(self):
        display(HTML(self.html_olustur()))

    def dosyaya_kaydet(self, dosya_adi="tarif_defteri.html"):
        # istenirse sayfa tek başına açılabilen bir html dosyası olarak da kaydedilebiliyor
        with open(dosya_adi, "w", encoding="utf-8") as dosya:
            dosya.write("<!DOCTYPE html><html lang='tr'><head><meta charset='utf-8'>"
                        "<meta name='viewport' content='width=device-width, initial-scale=1'>"
                        "<title>Tarif Defteri</title></head><body style='margin:0'>"
                        + self.html_olustur() + "</body></html>")
        print(dosya_adi, "kaydedildi.")

## 10. Tarif defterini çalıştırma

In [ ]:
arayuz = Arayuz(defter)
arayuz.goster()

İstersen sayfayı tek başına tarayıcıda açılabilen bir HTML dosyası olarak da kaydedebilirsin:

In [ ]:
arayuz.dosyaya_kaydet("tarif_defteri.html")